# expdpy — Explore panel data

_Notebook version: built 2026-07-01 02:38 UTC — re-open this notebook from GitHub if yours is older, to get the latest version._

A cloud-runnable walkthrough of the **Explore** module of [expdpy](https://github.com/cmg777/expdpy) on the bundled `kuznets` panel. Run the install cell below first, then run the rest top to bottom.

> The first cell installs everything and then **restarts the Colab runtime once** so the upgraded NumPy loads cleanly. When it reconnects, run the cells again (Runtime > Run all) — the install cell skips the restart the second time.

This notebook mirrors the [Explore page](https://cmg777.github.io/expdpy/explore.html) of the docs.

In [ ]:
import importlib.util
import os

!pip install -q "numpy>=2.1" "numba>=0.61" "expdpy @ git+https://github.com/cmg777/expdpy.git"
!pip install -q --force-reinstall --no-deps "expdpy @ git+https://github.com/cmg777/expdpy.git"

_RESTART_FLAG = "/tmp/.expdpy_runtime_restarted"
_ON_COLAB = importlib.util.find_spec("google.colab") is not None
if _ON_COLAB and not os.path.exists(_RESTART_FLAG):
    with open(_RESTART_FLAG, "w"):
        pass
    print("Install complete - restarting the runtime once so NumPy loads cleanly.")
    print("After it reconnects, run the cells again (Runtime > Run all).")
    os.kill(os.getpid(), 9)

In [ ]:
# Ensure Plotly figures render in Google Colab (a no-op in other notebook frontends).
import plotly.io as pio

try:
    import google.colab  # noqa: F401  (present only on Colab)

    pio.renderers.default = "colab"
except ImportError:
    pass

The **Explore** module is your first look at a dataset — before you fit a single model. This
page is a **case study**: you have just been handed a country–year panel and asked whether
inequality follows the famous **Kuznets curve** in income — rising, then falling, as economies
develop — and what *determines* it. We will answer that the way an analyst actually works:
**explore first**, model later.

The lead dataset is the bundled `kuznets` panel (80 countries observed every year over
2015–2025). Its regional inequality measure (`gini_regional`) is engineered to trace an
**N-shaped** Kuznets pattern in (log) GDP per capita, surrounded by a realistic set of
*determinants* — trade openness, resource rents, democracy, schooling, FDI, population.

Every Explore function takes a `pandas` DataFrame and returns a small **result object** carrying
a tidy `.df` plus an interactive [Plotly](https://plotly.com/python/) figure (`.fig`) or a
publication-quality [Great Tables](https://posit-dev.github.io/great-tables/) object (`.gt`).
Many also offer a plain-language `.interpret()`. Read this page top to bottom: the functions are
ordered as a **workflow** — *know the panel → describe it → split its variation → trace its
trends → compare groups → study relationships → measure its dynamics*.

::: {.callout-note}
This is **exploratory** analysis: every reading below describes an *association*, never a cause.
The [Analyze](analyze.qmd) module turns these patterns into estimates, and [Learn](learn.qmd)
explains the ideas behind them.
:::

## Stage 0 — Set up the panel

A panel has two coordinates: an **entity** (here the country) and a **time** index (the year).
Declaring them once means every call below can omit them, and panel-aware tables know to
summarize *by period*. `set_labels` does double duty: it attaches the data dictionary's
human-readable labels (so figures say "Regional inequality (Gini)" instead of `gini_regional`),
and with `set_panel=True` it also reads the `entity` / `time` tags from that dictionary and
declares the panel.

In [ ]:
import expdpy as ex
from expdpy.data import load_kuznets, load_kuznets_data_def

df = ex.set_labels(load_kuznets(), load_kuznets_data_def(), set_panel=True)
df[["country", "continent", "year", "gini_regional", "gdp_pc", "log_gdp_pc"]].head()

You can also declare the panel directly with `set_panel`, and read back what is stored with
`resolve_panel` — handy when a step (a merge, a column subset) drops the metadata and you need to
re-declare it:

In [ ]:
ex.set_panel(df, entity="country", time="year")
ex.resolve_panel(df)  # -> ('country', 'year')

## Stage 1 — Know the panel's skeleton

Before looking at a single value, learn the *shape* of the data. Is the panel balanced? Are
there gaps? Where is information missing? A surprising amount of analysis goes wrong because this
step was skipped.

`explore_panel_structure` summarizes balance and coverage and draws a **presence grid** (one row
per country, one column per year): solid means observed, blank means missing.

In [ ]:
structure = ex.explore_panel_structure(df)
structure.gt

In [ ]:
structure.fig

Some variables are missing far more often than others. `explore_missing_values_plot` maps the
fraction missing by variable and year — notice the determinants (`resource_rents`, `polity2`,
`school_enrollment`, `gasoline_price`) are patchier, and heavier in the early years. Knowing this
*now* prevents a model later from silently dropping half your sample.

In [ ]:
ex.explore_missing_values_plot(df).fig

`explore_value_heatmap` shows the raw outcome across the whole country-by-year grid. Standardizing
**by time** (a z-score within each year) strips out the common trend so that what remains is each
country's *relative* position — who is persistently more unequal than its peers.

In [ ]:
ex.explore_value_heatmap(df, var="gini_regional", standardize="by_time").fig

## Stage 2 — Clean and describe the variables

Now meet the variables one at a time: their level, spread, and shape.

Several determinants are right-skewed (GDP per capita, resource rents, aid): a handful of huge
values can dominate a correlation or a scatter. `treat_outliers` **winsorizes** — it caps values
at the 1st/99th percentile rather than dropping rows. We build a cleaned `analysis` frame here and
reuse it for the relationship views in Stage 6.

In [ ]:
cols = [
    "gini_regional",
    "gdp_pc",
    "log_gdp_pc",
    "trade_share",
    "resource_rents",
    "school_enrollment",
    "polity2",
]
analysis = ex.set_labels(
    ex.treat_outliers(df[cols], percentile=0.01), load_kuznets_data_def()
)
analysis.describe().round(2)

`explore_descriptive_table` reports the standard statistics for every numeric variable. Because
the panel is declared, it lays them out **by period** — here the first and last year — so you can
read level *and* change at a glance. The note beneath records the panel's dimensions and which
variables carry missing data.

In [ ]:
ex.explore_descriptive_table(df, periods=[2015, 2025]).gt

`explore_histogram` shows the distribution of one variable. The outcome `gini_regional` is fairly
symmetric; switch the variable to `gdp_pc` and you will see the long right tail that motivates the
**log** transform used throughout the Kuznets literature (and provided here as `log_gdp_pc`).

In [ ]:
ex.explore_histogram(df, "gini_regional", kde=True).fig

`explore_bar_plot` counts the levels of a categorical variable — how many observations fall in
each continent.

In [ ]:
ex.explore_bar_plot(df, "continent").fig

`explore_ext_obs_table` lists the most extreme observations — the country-years with the highest
and lowest inequality. Extremes are where data errors hide, and where the substantive story is
often most vivid.

In [ ]:
ex.explore_ext_obs_table(
    df, n=5, var="gini_regional", entity=["country"], time="year"
).gt

## Stage 3 — Two kinds of variation: within vs between

This is the idea that makes panel data special. Every variable varies in **two** directions: across
countries (*between*) and over time within a country (*within*). Mixing them up is the classic
panel mistake, so it pays to separate them up front.

`explore_xtsum_table` decomposes each variable's variation into overall, between, and within
components (the Stata `xtsum` you may know). If almost all of a variable's variation is *between*
countries, then over an 11-year window it is nearly a fixed trait; if much is *within*, it genuinely
moves over time.

In [ ]:
xt = ex.explore_xtsum_table(df, var=["gini_regional", "log_gdp_pc", "trade_share"])
xt.gt

In [ ]:
print(xt.interpret())

`explore_spaghetti_plot` makes the within variation visible: one faint line per country plus a bold
mean overlay. Here are the development trajectories — most countries drift upward together, which is
exactly the kind of common time trend a panel model will later absorb.

In [ ]:
ex.explore_spaghetti_plot(df, var="log_gdp_pc").fig

## Stage 4 — Trends over time

How does inequality evolve for the panel as a whole?

`explore_trend_plot` plots the cross-country mean each year with standard-error bars — the average
trajectory.

In [ ]:
ex.explore_trend_plot(df, var=["gini_regional"]).fig

A mean can hide what happens in the tails. `explore_quantile_trend_plot` tracks several quantiles
at once, so you can see whether the *spread* of inequality across countries is widening or
narrowing over time.

In [ ]:
ex.explore_quantile_trend_plot(df, var="gini_regional").fig

`explore_distribution_over_time` shows the *whole* distribution shifting year by year as a
**ridgeline** — each ridge is one year's density. Is the distribution simply sliding, or is it
also changing shape (skew, bimodality)?

In [ ]:
ex.explore_distribution_over_time(df, var="gini_regional").fig

## Stage 5 — Compare groups

The panel groups countries by continent (and flags federal states). Comparing across these groups
is often the first hint of *what* drives the outcome.

`explore_bar_plot_by_group` reduces each group to a single statistic — here the mean level of
inequality per continent.

In [ ]:
ex.explore_bar_plot_by_group(df, "continent", "gini_regional").fig

A bar hides the spread inside each group. `explore_violin_plot_by_group` draws the full
distribution per continent, so you can see overlap and outliers a mean would mask.

In [ ]:
ex.explore_violin_plot_by_group(df, "continent", "gini_regional").fig

`explore_box_plot` gives the same comparison as boxes — and because we have a panel, passing
`time=` turns it into an animation: press the **▶** button (bottom-left, beside the year slider) to
watch each continent's distribution move year by year, and drag the slider to pause on a year. The
value axis stays pinned so the motion is comparable. Extra arguments tune the view — here we order
the continents by their mean and drop the whisker points:

In [ ]:
ex.explore_box_plot(
    df, "continent", "gini_regional", time="year", order_by_mean=True, points=False
).fig

Every `explore_*` distribution result reads itself back in plain language — `.interpret()` narrates
how the pooled median and the between-group gap shifted:

In [ ]:
print(ex.explore_box_plot(df, "continent", "gini_regional", time="year").interpret())

`explore_strip_plot` shows every region as a point instead of a summary — useful when a group has
only a handful of members. **Hover a point** and its unit's name appears in the info box. It
animates over time the same way (large panels are thinned to `max_units` points to stay smooth):

In [ ]:
ex.explore_strip_plot(df, "continent", "gini_regional", time="year", alpha=0.5).fig

`explore_trend_plot_by_group` puts time back in: one trend line per continent, revealing whether the
groups move together or diverge.

In [ ]:
ex.explore_trend_plot_by_group(df, group_var="continent", var="gini_regional").fig

### Composition — who makes up the whole, and how it shifts

Group comparisons ask *how big* each group is on average; composition views ask *what share* of a
total each part holds. `explore_treemap_plot` nests countries within continents, sizes each tile by
population and colors it by inequality, and — with `time=` — animates the whole picture across the
years (same **▶** control, bottom-left). The color scale is held fixed across frames so the shifts
are comparable.

In [ ]:
ex.explore_treemap_plot(
    df,
    path=["continent", "country"],
    size="population",
    color="gini_regional",
    time="year",
).fig

`explore_sunburst_plot` is the radial counterpart: the same hierarchy as concentric rings, so nested
part-to-whole shares read at a glance.

In [ ]:
ex.explore_sunburst_plot(
    df,
    path=["continent", "country"],
    size="population",
    color="gini_regional",
    time="year",
).fig

## Stage 6 — Relationships and the Kuznets curve

Now the central question. We work on the winsorized `analysis` frame so a few extreme values do not
distort the picture.

`explore_correlation_table` reports pairwise associations — Pearson above the diagonal, Spearman
(rank) below, significant cells in bold. This is the quickest scan of which determinants co-move with
inequality.

In [ ]:
ex.explore_correlation_table(analysis).gt

`explore_correlation_plot` shows the same matrix as a heatmap (an `ellipse` style is also available)
— easier to spot blocks of related variables at a glance.

In [ ]:
ex.explore_correlation_plot(analysis).fig

And here it is — the **N-shaped Kuznets curve**. `explore_scatter_plot` puts inequality against (log)
GDP per capita, colored by continent and sized by population, with a LOESS smoother that traces the
rise–fall–rise without assuming any functional form.

In [ ]:
ex.explore_scatter_plot(
    df, x="log_gdp_pc", y="gini_regional", color="continent", size="population", loess=1
).fig

But is that curve a story about *rich versus poor countries* (between) or about *a country getting
richer over time* (within)? They can even have opposite signs — Simpson's paradox for panels.
`explore_scatter_plot_within_between` splits the relationship into pooled, between, and within clouds,
each with its own fitted slope, so you can see which one the raw scatter was really showing.

In [ ]:
wb = ex.explore_scatter_plot_within_between(df, x="log_gdp_pc", y="gini_regional")
wb.fig

In [ ]:
print(wb.interpret())

## Stage 7 — Dynamics and persistence

A panel lets you ask a question a cross-section cannot: is a country's inequality **sticky**?

`explore_transition_matrix` bins inequality into states (here quartiles) and tabulates how often a
country moves between them from one year to the next. A heavy diagonal means high persistence — where
you start is where you stay.

In [ ]:
ex.explore_transition_matrix(df, var="gini_regional", n_bins=4).fig

`explore_within_persistence` measures the same idea continuously: this year's value against last
year's, within country, with the AR(1) serial-correlation `rho`. The closer `rho` is to 1, the more
inequality this year is just a near-copy of last year's.

In [ ]:
persistence = ex.explore_within_persistence(df, var="gini_regional")
persistence.fig

In [ ]:
print(persistence.interpret())

## Where to go next

You now have the full exploratory picture: a balanced panel with patchy determinants, an N-shaped
curve that is mostly a *between*-country pattern, and an inequality measure that is highly
persistent. The natural next step is to estimate it.

- [Analyze panel data](analyze.qmd) — fit the cubic Kuznets curve with **two-way fixed effects**
  and clustered standard errors, then compare pooled / between / within estimators.
- [Learn panel data](learn.qmd) — the ideas behind within vs between, fixed effects and
  demeaning, with runnable sandboxes.